In [0]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, isnull
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline

# 1. Load the Silver Delta table
SILVER_TABLE = "workspace.default.cybersecurity_training_silver"
GOLD_TABLE = "workspace.default.cybersecurity_training_gold"

df_silver = spark.table(SILVER_TABLE)
print("Silver table loaded: " + SILVER_TABLE)

# 2. Inspect and print the actual schema
print("--- Silver Schema ---")
df_silver.printSchema()

# Collect all column names and types from the actual schema
schema_fields = df_silver.schema.fields
all_columns = [field.name for field in schema_fields]
print("All columns: " + str(all_columns))

# 3. Identify the target column "label"
TARGET_COLUMN = "label"
if TARGET_COLUMN not in all_columns:
    raise Exception("Target column 'label' not found in the Silver table.")
print("Target column identified: " + TARGET_COLUMN)

# 4. Remove unnecessary identifier columns such as "id" if they exist
ID_COLUMNS_TO_DROP = ["id", "Id", "ID", "attack_cat", "attack_category"]
columns_to_drop = [c for c in ID_COLUMNS_TO_DROP if c in all_columns]
print("Columns to drop: " + str(columns_to_drop))
df_clean = df_silver.drop(*columns_to_drop)

# Re-collect columns after dropping
remaining_columns = [field.name for field in df_clean.schema.fields]
print("Remaining columns after drop: " + str(remaining_columns))

# 5. Keep useful numerical network traffic features
# Identify numerical columns (excluding the label)
numerical_columns = [
    field.name
    for field in df_clean.schema.fields
    if field.dataType.typeName() in ("integer", "long", "double", "float", "decimal", "short")
    and field.name != TARGET_COLUMN
]
print("Numerical columns: " + str(numerical_columns))

# 6. Keep useful categorical cybersecurity features such as proto, service, state if they exist
KNOWN_CATEGORICAL_CANDIDATES = ["proto", "service", "state", "proto_", "service_", "state_"]
categorical_columns = [
    field.name
    for field in df_clean.schema.fields
    if field.dataType.typeName() == "string"
    and field.name != TARGET_COLUMN
    and field.name in KNOWN_CATEGORICAL_CANDIDATES
]
print("Categorical columns: " + str(categorical_columns))

# 7. Handle NULL values safely
# Fill numerical NULLs with 0
if numerical_columns:
    fill_dict_numerical = {c: 0 for c in numerical_columns}
    df_clean = df_clean.fillna(fill_dict_numerical)

# Fill categorical NULLs with "unknown"
if categorical_columns:
    fill_dict_categorical = {c: "unknown" for c in categorical_columns}
    df_clean = df_clean.fillna(fill_dict_categorical)

# Fill label NULLs (if any) with 0 as a safe default
df_clean = df_clean.fillna({TARGET_COLUMN: 0})

print("NULL values handled.")

# Verify no remaining NULLs in the feature columns
null_check_columns = numerical_columns + categorical_columns
if null_check_columns:
    df_clean.select([
        count(when(isnull(c), c)).alias(c) for c in null_check_columns
    ]).show(truncate=False)

# 8. Convert categorical columns into numerical indexes using StringIndexer with handleInvalid="keep"
indexers = []
for c in categorical_columns:
    indexer = StringIndexer(
        inputCol=c,
        outputCol=c + "_idx",
        handleInvalid="keep"
    )
    indexers.append(indexer)

print("StringIndexer stages created for: " + str(categorical_columns))

# 9. Create a single ML feature vector using VectorAssembler
# Assembler input = all numerical columns + indexed categorical columns
assembler_inputs = numerical_columns + [c + "_idx" for c in categorical_columns]
print("Assembler inputs: " + str(assembler_inputs))

assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="features"
)

# Build a pipeline so indexers and assembler run in order
pipeline = Pipeline(stages=indexers + [assembler])

# Fit and transform
pipeline_model = pipeline.fit(df_clean)
df_gold = pipeline_model.transform(df_clean)

# 10. Create a final DataFrame containing only features and label
df_final = df_gold.select("features", TARGET_COLUMN).withColumnRenamed(TARGET_COLUMN, "label")

# Cast label to integer (suitable for classification)
from pyspark.sql.types import IntegerType
df_final = df_final.withColumn("label", col("label").cast(IntegerType()))

print("Final DataFrame created with columns: features, label")

# 11. The dataset is now suitable for Random Forest classification
# (features vector + integer label column)

# 12. Display the final schema, row count and sample records
print("--- Gold Final Schema ---")
df_final.printSchema()

print("--- Row Count ---")
total_rows = df_final.count()
print("Total rows: " + str(total_rows))

print("--- Sample Records ---")
df_final.show(10, truncate=False)

# 13. Save the final Gold dataset as Delta, overwrite mode
df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(GOLD_TABLE)

print("Gold table saved to: " + GOLD_TABLE)

# 14. Final confirmation
print("Gold layer created successfully")
print("Final number of records: " + str(total_rows))

Silver table loaded: workspace.default.cybersecurity_training_silver
--- Silver Schema ---
root
 |-- id: integer (nullable = true)
 |-- dur: double (nullable = true)
 |-- proto: string (nullable = true)
 |-- service: string (nullable = true)
 |-- state: string (nullable = true)
 |-- spkts: integer (nullable = true)
 |-- dpkts: integer (nullable = true)
 |-- sbytes: integer (nullable = true)
 |-- dbytes: integer (nullable = true)
 |-- rate: double (nullable = true)
 |-- sttl: integer (nullable = true)
 |-- dttl: integer (nullable = true)
 |-- sload: double (nullable = true)
 |-- dload: double (nullable = true)
 |-- sloss: integer (nullable = true)
 |-- dloss: integer (nullable = true)
 |-- sinpkt: double (nullable = true)
 |-- dinpkt: double (nullable = true)
 |-- sjit: double (nullable = true)
 |-- djit: double (nullable = true)
 |-- swin: integer (nullable = true)
 |-- stcpb: long (nullable = true)
 |-- dtcpb: long (nullable = true)
 |-- dwin: integer (nullable = true)
 |-- tcprtt: do